In [2]:
from utils import *
import numpy as np
import torch
from torch.nn.functional import silu

In [3]:
path = ROOT.parent / 'data' / 'invert' / 'from_val_rk4'

In [ ]:
class TrajDataset():
    """Point-wise loss, trajectory-wise batches.

    The (lead, ch, x, y) grid is the same for every trajectory, so it is built once
    (flattened to P = (L-1)*C*H*W points) and the model is evaluated once per step on
    that grid. z_prev / z_next are (n_traj, P) and the point-wise loss broadcasts over
    the trajectory dim. Same loss values as the pixel-shuffled version, ~20x faster."""
    def __init__(self, path, split, device, traj_per_batch=8):
        Z, _, _ = load_invert(path)
        N, L, C, H, W = Z.shape
        N = int(N*0.8)
        Z = Z[:N] if split == 'train' else Z[N:]
        self.Z = torch.from_numpy(Z).to(device)                       # (n, L, C, H, W)
        self.n, self.L, self.C, self.H, self.W = self.Z.shape
        self.P = (L - 1) * C * H * W
        self.size = self.n * self.P                                     # number of loss terms per epoch
        self.traj_per_batch = traj_per_batch

        t, ch, x, y = torch.meshgrid(torch.arange(L - 1), torch.arange(C), torch.arange(H), torch.arange(W), indexing='ij')
        self.lead = (t.reshape(-1) + 2).float().to(device)              # Z index t+1 is lead t+2
        self.ch = ch.reshape(-1).to(device)
        self.x = x.reshape(-1).float().to(device)
        self.y = y.reshape(-1).float().to(device)

    def iterate_batches(self):
        perm = torch.randperm(self.n, device=self.Z.device)
        for start in range(0, self.n, self.traj_per_batch):
            idx = perm[start:start + self.traj_per_batch]
            z = self.Z[idx]                                             # (b, L, C, H, W)
            yield self.lead, self.ch, self.x, self.y, z[:, :-1].reshape(len(idx), -1), z[:, 1:].reshape(len(idx), -1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_ds = TrajDataset(path, split='train', device=device)
# test code
lead, ch, x, y, z_prev, z_next = next(train_ds.iterate_batches())
print(train_ds.n, 'trajectories,', train_ds.P, 'points each')
print(lead.shape, ch.shape, x.shape, y.shape, z_prev.shape, z_next.shape)
print(f"x max: {torch.max(x)}, x min {torch.min(x)}")
print(f"y max: {torch.max(y)}, y min {torch.min(y)}")
print(f"lead max: {torch.max(lead)}, lead min {torch.min(lead)}")


In [ ]:
val_ds = TrajDataset(path, split='val', device=device)
print(val_ds.n, 'trajectories,', val_ds.size, 'loss terms')


In [ ]:
class PositionalEmbedding(torch.nn.Module):
    def __init__(self, num_channels, max_positions=24, endpoint=False):
        super().__init__()
        self.num_channels = num_channels
        self.max_positions = max_positions
        self.endpoint = endpoint

    def forward(self, x):
        #FROMTRACE:
        # num_channels decide how long the output will be
        # max_position decides the slowest wave
        
        # Define the frequencies domain from [0, num_channels//2]
        # Normalise the frequencies domain to [0, 1]
        # For each f: x = x * max_positions^(-f)
        freqs = torch.arange(start=0, end=self.num_channels //
                             2, dtype=torch.float32, device=x.device)
        freqs = freqs / (self.num_channels // 2 - (1 if self.endpoint else 0))
        freqs = (1 / self.max_positions) ** freqs
        x = x.ger(freqs.to(x.dtype))
        x = torch.cat([x.cos(), x.sin()], dim=1)
        return x

class FourierEmbedding(torch.nn.Module):
    # num_channels = 0 gives an empty embedding (coordinate ignored)
    def __init__(self, num_channels, period=64):
        super().__init__()
        n_freq = num_channels // 2                    # F = num_channels / 2
        freqs = torch.arange(1, n_freq + 1, dtype=torch.float32) / period
        self.register_buffer('freqs', freqs)   

    def forward(self, x):
        # Outer product
        x = x.ger((2 * np.pi * self.freqs).to(x.dtype))
        x = torch.cat([x.cos(), x.sin()], dim=1)
        return x

class TimeVaryingOU(torch.nn.Module):
    # embedding t with positional embedding
    # embedding x / y with fourier embedding (x_dim / y_dim channels; 0 = ignore that coordinate)
    # embedding ch with nn.Embedding(2, d)
    # n_hidden MLP layers with SiLU, softplus heads
    # tuned: 1 hidden layer, no x, one y mode (val nll 0.1157 vs 0.1160 for the 3-layer xy_dim=8 model)
    def __init__(self, emb_dim=64, hidden_dim=256, n_hidden=1, x_dim=0, y_dim=2):
        super().__init__()
        self.t_emb = PositionalEmbedding(emb_dim)
        self.x_emb = FourierEmbedding(x_dim)
        self.y_emb = FourierEmbedding(y_dim)
        self.ch_emb = torch.nn.Embedding(2, emb_dim)

        in_dim = emb_dim + x_dim + y_dim + emb_dim 
        dims = [in_dim] + [hidden_dim] * n_hidden
        self.layers = torch.nn.ModuleList(
            torch.nn.Linear(d_in, d_out) for d_in, d_out in zip(dims[:-1], dims[1:])
        )
        self.out = torch.nn.Linear(hidden_dim, 2)
        torch.nn.init.zeros_(self.out.weight)
        self.out.bias.data.copy_(torch.tensor([0.541, 0.07]))   # sigma = 1, alpha ≈ 0.7

    def forward(self, t, x, y, ch):
        embedded_x = self.x_emb(x)
        embedded_y = self.y_emb(y)
        embedded_ch = self.ch_emb(ch)
        embedded_t = self.t_emb(t)
        h = torch.cat([embedded_t, embedded_x, embedded_y, embedded_ch], dim=1)
        for layer in self.layers:
            h = silu(layer(h))
        h = self.out(h)
        sigma = torch.nn.functional.softplus(h[:, 0]) + 1e-6
        c2 = torch.nn.functional.softplus(h[:, 1]) + 1e-6
        return sigma, c2


class TableTiltOU(torch.nn.Module):
    """Smallest model the data support: a free value per (lead, channel) plus a linear
    'tilt' on the lowest Fourier modes of y (and optionally x).
        pre-softplus [sigma, c2] = tab[k, ch] + W[k, ch] @ [cos(2πy/64), sin(2πy/64), ...]
    288 parameters, trains in ~1 min, val nll 0.1156-0.1157 (best of everything tried)."""
    def __init__(self, n_leads=24, x_dim=0, y_dim=2, period=64):
        super().__init__()
        self.x_emb = FourierEmbedding(x_dim, period=period)
        self.y_emb = FourierEmbedding(y_dim, period=period)
        self.nb = x_dim + y_dim
        self.tab = torch.nn.Embedding(n_leads * 2, 2)
        self.tab.weight.data.copy_(torch.tensor([0.541, 0.07]).expand(n_leads * 2, 2))   # sigma = 1, alpha ≈ 0.7
        self.tilt = torch.nn.Embedding(n_leads * 2, 2 * self.nb)
        torch.nn.init.zeros_(self.tilt.weight)

    def forward(self, t, x, y, ch):
        idx = (t.long() - 1) * 2 + ch                                               # lead 1..24 -> row
        basis = torch.cat([self.x_emb(x), self.y_emb(y)], dim=1)                    # (B, nb)
        W = self.tilt(idx).view(-1, 2, self.nb)                                     # (B, 2, nb)
        h = self.tab(idx) + torch.einsum('bon,bn->bo', W, basis)
        sigma = torch.nn.functional.softplus(h[:, 0]) + 1e-6
        c2 = torch.nn.functional.softplus(h[:, 1]) + 1e-6
        return sigma, c2


In [ ]:
def batch_loss(batch):
    lead, ch, x, y, z_prev, z_next = (b.to(device) for b in batch)
    sigma_next, c2 = model(lead, x, y, ch)          # sigma at lead k, c^2 on interval [k-1, k]
    sigma_prev, _ = model(lead - 1, x, y, ch)       # sigma at lead k-1
    alpha = torch.exp(-0.5 * c2)                    # dt = 1 h
    one_m_a2 = -torch.expm1(-c2)                    # 1 - alpha^2, stable as alpha -> 1
    xi_prev, xi_next = z_prev / sigma_prev, z_next / sigma_next   # (n_traj, P) / (P,)
    loss = 0.5 * (torch.log(sigma_next**2) + torch.log(one_m_a2) + (xi_next - alpha * xi_prev)**2 / one_m_a2)
    first = lead == 2                               # transition 1 -> 2 also carries the lead-1 marginal
    loss = loss + first * 0.5 * (torch.log(sigma_prev**2) + xi_prev**2)
    return loss.mean(), loss.numel()


In [ ]:
def train(model, train_ds, val_ds, device, epochs=200, lr=3e-3, weight_decay=1e-4, print_every=10):
    # dt is the gap between consecutive leads in hours, so c is in physical units
    # one epoch = n_traj / traj_per_batch optimizer steps (15 for the train split), hence the many epochs
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    history = {'train': [], 'val': []}
    best = (float('inf'), None)
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for batch in train_ds.iterate_batches():
            loss, n = batch_loss(batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * n
        history['train'].append(total / train_ds.size)

        model.eval()
        total = 0.0
        with torch.no_grad():
            for batch in val_ds.iterate_batches():
                loss, n = batch_loss(batch)
                total += loss.item() * n
        history['val'].append(total / val_ds.size)
        if history['val'][-1] < best[0]:
            best = (history['val'][-1], {k: v.detach().clone() for k, v in model.state_dict().items()})

        if (epoch + 1) % print_every == 0 or epoch == 0 or epoch == epochs - 1:
            print(f"epoch {epoch + 1}/{epochs}  train nll {history['train'][-1]:.4f}  val nll {history['val'][-1]:.4f}")
    print(f"best val nll {best[0]:.5f} at epoch {int(np.argmin(history['val'])) + 1}, last val nll {history['val'][-1]:.5f}")
    history['best_state'] = best[1]
    return history


model = TimeVaryingOU()                     # tuned MLP (1 hidden layer, no x, one y mode): val nll ≈ 0.1157
# model = TableTiltOU()                     # table + y-tilt alternative: val nll ≈ 0.1157 (0.1156 at best epoch), use lr=3e-3, epochs=200
history = train(model, train_ds, val_ds, device, epochs=150, lr=3e-4)


### Tuning notes (2026-11-12)

Val nll of the notebook loss on the 30 held-out trajectories. Reference: original 3-layer MLP, `xy_dim=8`, lr 1e-4 → **0.1160**; 1 hidden layer, lr 1e-5 → **0.1159**.

**Closed-form floors** (fit on train, evaluated on val): per-pixel MLE 0.1284 (massive over-fit); per `(lead, ch)` only, no spatial dependence, **0.1159** (0.1158 even when fitted on val itself). So almost all spatial detail is trajectory noise.

| model | val nll (best / last) |
|---|---|
| MLP, 1 hidden, no x,y | 0.1159 / 0.1159 |
| lead×channel table (96 params) | 0.1158 / 0.1159 |
| MLP, 1 hidden, `xy_dim=2` (one wave in x and y) | 0.1157 / 0.1157 (3 seeds agree) |
| MLP, 1 hidden, x wave only | 0.1158 / 0.1159 |
| MLP, 1 hidden, y wave only | 0.1157 / 0.1157 |
| `xy_dim=4` or two y modes | 0.1158 / 0.1159 |
| **table + y-tilt (`TableTiltOU`, 288 params)** | **0.1156 / 0.1157** |

Takeaways: only the single lowest Fourier mode in **y** transfers to val; x carries nothing; weight decay, EMA, width, depth, FiLM conditioning on lead make no difference at the 1e-4 level. Everything converged sits within 1e-4 of the floor of its family, so the remaining gap to train nll (0.1097) is a train/val distribution difference, not something a bigger model can close.
